# Part 10, notebook 3 of 5: one sub-agent per box, and a panel

Companion to *When One Agent Is Not Enough*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 10a | `10a_three_ways` | Three ways to spend model contexts | two short live cells |
| 10b | `10b_supervisor_and_handoff` | A supervisor and two specialists | live cells (about 1 minute) |
| 10c | `10c_fanout_and_panel` | One sub-agent per box, and a panel | live cells (about 30 seconds) |
| 10d | `10d_the_200_image_results` | The results | no (reads the saved runs) |
| 10e | `10e_failures_and_tests` | What goes wrong | no |

**The picture.** Marking a pile of exams. One teacher marks all 40 papers in a row and gets tired and inconsistent by the end. Or you split the pile: each marker gets one paper
and a mark scheme, and none of them has seen the others. They work at the same time and a tired marker can only spoil one paper.
A **panel** is the same paper given to three markers, with the majority winning.

In [1]:
import sys, json, os
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))     # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
os.environ.setdefault("PART10_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves
import sub_agents as sa
import sub_summary as ss
%matplotlib inline

## See exactly what each sub-agent was shown

The cell wraps `sa.chat` so it records the messages of every call, then runs the three Q2 arms on one image. Look at how small each fan-out context is, and that the
box it holds is the only box it can see.

**Your turn:** change `IMAGE`.

In [2]:
IMAGE = "test_00046"
seen = []
orig = sa.chat
def recording_chat(messages, meter, role, tools=None, temperature=sa.TEMPERATURE):
    seen.append((role, temperature, [m["content"] if isinstance(m, dict) else m for m in messages]))
    return orig(messages, meter, role, tools, temperature)
sa.chat = recording_chat
try:
    out = {name: sa.ARMS[name](IMAGE) for name in ("batched", "fanout")}
    out["panel3"] = sa.arm_panel3(IMAGE, workers=3)   # 3 at a time: the shared server crashed twice under 9 concurrent requests
finally:
    sa.chat = orig

for name, r in out.items():
    print(f"{name:8s} boxes={r['n_candidates']} kept={r['kept']} tp/fp/fn={r['tp']}/{r['fp']}/{r['fn']} calls={r['calls']} "
          f"tokens={r['prompt_tokens'] + r['completion_tokens']} biggest context={r['peak_prompt']} judge failures={r['judge_failed']}")

batched  boxes=4 kept=[0] tp/fp/fn=1/0/1 calls=1 tokens=664 biggest context=414 judge failures=0
fanout   boxes=4 kept=[0] tp/fp/fn=1/0/1 calls=4 tokens=1251 biggest context=259 judge failures=0
panel3   boxes=4 kept=[0] tp/fp/fn=1/0/1 calls=12 tokens=3747 biggest context=259 judge failures=0


In [3]:
one_box = [m for role, t, m in seen if len(m) == 2 and m[1].startswith("Candidate:")]
print("a fan-out sub-agent's whole world (system prompt shortened):\n")
print(one_box[0][0][:260] + " ...\n")
print(one_box[0][1])
print(f"\n{len(one_box)} such contexts were created for this image (fanout + panel3 together)")

a fan-out sub-agent's whole world (system prompt shortened):

You judge ONE candidate box from a noisy classical detector on an X-ray image of a metal part. You get its
measurement record and no image. Most candidates are false alarms.
Choose a label: "defect", "false_alarm" or "uncertain", and a confidence from 0 to 1.
 ...

Candidate:
{"candidate_id": 0, "contrast": 30.29, "area_px": 483, "aspect_ratio": 0.91, "edge_distance_px": 71}

16 such contexts were created for this image (fanout + panel3 together)


## Why the panel runs warm

Three copies at temperature 0 would give the same answer three times. The panel runs at temperature 0.7. The price is that sampling more freely also produces more malformed answers.
The saved 200-image runs say how many:

In [4]:
s = ss.summarize()
print("panel3 sub-agent calls that failed twice:", s["arms"]["panel3"]["pooled"]["judge_failed"], "of", s["panel_boxes"] * 3, "judgements")
print("boxes where the three votes were not unanimous:", s["panel_split_boxes"], "of", s["panel_boxes"])
print("fanout (temperature 0) failures:", s["arms"]["fanout"]["pooled"]["judge_failed"])

panel3 sub-agent calls that failed twice: 77 of 3078 judgements
boxes where the three votes were not unanimous: 221 of 1026
fanout (temperature 0) failures: 9
